# 💬 Interface de Chat com Qwen - Fenômenos de Transporte

Interface interativa para conversar com a IA Qwen sobre conteúdos do livro.

## 🔐 Configuração Inicial

In [ ]:
# Importações
import os
import sys
from pathlib import Path
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets

sys.path.insert(0, str(Path.cwd()))
from qwen_integration.qwen_client import QwenClient, CHAPTERS

In [ ]:
# Verifica API Key
api_key = os.getenv("QWEN_API_KEY")
if not api_key:
    display(Markdown("""
    ⚠️ **QWEN_API_KEY não configurada**
    
    Para habilitar o chat com IA:
    1. Acesse https://dashscope.aliyun.com/
    2. Crie uma conta e gere uma API Key
    3. Execute em seu terminal:
    ```bash
    export QWEN_API_KEY="sua-chave-aqui"
    ```
    
    *Enquanto isso, o modo offline está ativo com respostas pré-definidas.*
    """))

# Inicializa cliente
client = QwenClient(offline_mode=not api_key)

In [ ]:
# Interface de seleção
chapter_dropdown = widgets.Dropdown(
    options=[(f"{info['title']} (Vol {info['volume']})", cid) 
             for cid, info in CHAPTERS.items()],
    description='📚 Capítulo:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

level_toggle = widgets.ToggleButtons(
    options=['graduation', 'postgrad'],
    description='🎓 Nível:',
    button_style='info'
)

template_dropdown = widgets.Dropdown(
    options=[
        ('💡 Explicar conceito', 'explicacao'),
        ('✏️ Resolver exercício', 'exercicio'),
        ('💻 Gerar código', 'codigo')
    ],
    description='🔧 Tipo:',
    style={'description_width': 'initial'}
)

question_text = widgets.Textarea(
    placeholder='Digite sua pergunta sobre o capítulo selecionado...',
    description='❓ Pergunta:',
    layout=widgets.Layout(width='600px', height='80px')
)

send_button = widgets.Button(
    description='🚀 Enviar',
    button_style='success',
    icon='send'
)

clear_button = widgets.Button(
    description='🗑️ Limpar',
    button_style='warning'
)

output_area = widgets.Output(layout={'border': '1px solid #ccc', 'padding': '10px'})

ui = widgets.VBox([
    widgets.HBox([chapter_dropdown, level_toggle]),
    template_dropdown,
    question_text,
    widgets.HBox([send_button, clear_button]),
    output_area
])

display(ui)

In [ ]:
# Lógica de envio
def on_send(b):
    with output_area:
        output_area.clear_output()
        
        chapter = chapter_dropdown.value
        level = level_toggle.value
        template = template_dropdown.value
        question = question_text.value.strip()
        
        if not question:
            display(Markdown("⚠️ Por favor, digite uma pergunta."))
            return
        
        topic = CHAPTERS[chapter]['title']
        display(Markdown(f"**Pensando sobre {topic}...** 🤔"))
        
        try:
            resposta = client.ask(
                chapter=chapter,
                topic=topic,
                question=question,
                level=level,
                template=template
            )
            display(Markdown(resposta))
        except Exception as e:
            display(Markdown(f"❌ Erro: {str(e)}"))

def on_clear(b):
    question_text.value = ''
    with output_area:
        output_area.clear_output()

send_button.on_click(on_send)
clear_button.on_click(on_clear)

## 📌 Dicas de Uso

- **Graduação**: Respostas focadas em conceitos e aplicações práticas
- **Pós-Graduação**: Inclui derivações, análise dimensional, limitações dos modelos
- **Código**: Gera Python com numpy/scipy, pronto para executar
- **Cache**: Respostas idênticas são cacheadas por 24h para economizar API

## 🔗 Links Úteis

- [Documentação DashScope](https://help.aliyun.com/zh/dashscope/)
- [Modelos Qwen](https://help.aliyun.com/zh/dashscope/developer-reference/quick-start)
- [Repositório do Livro](https://github.com/JaderLugon/fenomenos-transporte-livro)